# abalone19: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('07_abalone19_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,3.392,71.846,0.000
smote,61.121,5.521,2.957,42.143,89.280,65.712,3.332,77.197,33.321
bsmote,26.425,4.153,2.634,12.381,95.654,54.017,4.410,76.922,15.289
ros,66.241,6.012,3.203,50.000,87.892,68.946,3.755,77.591,36.127
adasyn,62.288,5.623,3.007,43.810,89.087,66.448,3.292,76.921,33.956
casb,63.842,5.634,3.001,46.667,88.001,67.334,3.416,77.792,34.738
casb_relaxed,62.294,5.652,3.024,43.810,89.099,66.454,3.332,77.116,33.973
casb_edited,67.294,6.179,3.288,51.429,88.218,69.823,3.280,77.408,36.736
casb_ros,66.243,6.313,3.374,49.762,88.593,69.177,3.570,78.174,36.278


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,2.129,56.764,0.000
smote,38.954,4.058,2.241,21.667,92.612,57.139,1.945,65.882,21.506
bsmote,22.917,3.599,2.255,9.048,97.296,53.172,2.433,59.151,13.258
ros,22.966,4.062,2.650,9.048,97.368,53.208,1.676,56.813,13.514
adasyn,40.276,4.154,2.286,22.857,92.298,57.578,1.928,65.868,22.215
casb,33.865,4.791,2.844,15.238,95.896,55.567,1.697,57.315,19.328
casb_relaxed,38.985,4.128,2.284,21.667,92.781,57.224,1.914,65.287,21.557
casb_edited,43.848,4.259,2.311,27.381,91.127,59.254,1.964,66.676,24.053
casb_ros,22.966,4.062,2.650,9.048,97.368,53.208,1.676,56.813,13.514


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,3.350,69.145,0.000
smote,12.153,3.186,2.407,5.000,98.938,51.969,3.647,75.286,7.670
bsmote,0.000,0.000,0.000,0.000,99.445,49.722,3.069,71.900,0.000
ros,8.162,5.357,15.000,3.333,99.952,51.643,5.763,68.913,6.760
adasyn,12.173,3.538,2.742,5.000,98.901,51.951,3.015,74.759,7.855
casb,12.225,6.040,7.833,5.000,99.698,52.349,4.724,70.787,9.133
casb_relaxed,23.737,6.813,5.380,9.524,98.962,54.243,3.418,74.909,15.275
casb_edited,19.857,3.657,2.382,8.095,97.694,52.895,3.523,76.782,11.757
casb_ros,8.165,5.714,20.000,3.333,99.976,51.655,5.675,68.995,6.940


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,99.976,49.988,4.432,81.250,0.000
smote,67.957,4.981,2.608,57.143,83.413,70.278,4.483,82.541,36.469
bsmote,46.460,4.140,2.214,32.381,90.463,61.422,4.015,73.116,25.300
ros,69.859,4.950,2.587,60.714,82.134,71.424,4.408,83.701,37.404
adasyn,67.914,4.951,2.591,57.143,83.316,70.230,4.467,82.618,36.432
casb,67.317,4.989,2.615,57.381,83.522,70.452,4.625,83.336,36.153
casb_relaxed,65.543,4.762,2.496,54.048,83.727,68.887,4.435,82.193,35.152
casb_edited,67.664,4.807,2.512,57.381,82.810,70.095,4.682,82.439,36.236
casb_ros,68.759,5.092,2.667,59.286,83.051,71.168,4.738,83.376,36.925


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,99.994,49.997,3.326,69.751,0.000
smote,45.046,4.437,2.553,31.488,91.061,61.274,3.352,75.226,24.741
bsmote,23.950,2.973,1.776,13.452,95.714,54.583,3.482,70.272,13.462
ros,41.807,5.095,5.860,30.774,91.837,61.305,3.901,71.754,23.451
adasyn,45.663,4.566,2.657,32.202,90.901,61.552,3.176,75.041,25.115
casb,44.312,5.364,4.073,31.071,91.779,61.425,3.616,72.307,24.838
casb_relaxed,47.640,5.339,3.296,32.262,91.142,61.702,3.275,74.876,26.489
casb_edited,49.666,4.726,2.624,36.071,89.962,63.017,3.362,75.826,27.196
casb_ros,41.533,5.295,7.173,30.357,92.247,61.302,3.915,71.839,23.414


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,67.294,ros,66.241,1.052,WIN
1,f1,casb_ros,6.313,ros,6.012,0.301,WIN
2,precision,casb_ros,3.374,ros,3.203,0.171,WIN
3,recall,casb_edited,51.429,ros,50.000,1.429,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,43.848,adasyn,40.276,3.571,WIN
1,f1,casb,4.791,adasyn,4.154,0.637,WIN
2,precision,casb,2.844,ros,2.650,0.195,WIN
3,recall,casb_edited,27.381,adasyn,22.857,4.524,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,23.737,adasyn,12.173,11.564,WIN
1,f1,casb_relaxed,6.813,ros,5.357,1.456,WIN
2,precision,casb_ros,20.000,ros,15.000,5.000,WIN
3,recall,casb_relaxed,9.524,smote,5.000,4.524,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,49.666,adasyn,45.663,4.003,WIN
1,f1,casb,5.364,ros,5.095,0.268,WIN
2,precision,casb_ros,7.173,ros,5.860,1.313,WIN
3,recall,casb_edited,36.071,adasyn,32.202,3.869,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.040408,0.813028,0.000000,None
1,42,1,svm,smote,0.50,0.553165,0.052632,0.028571,0.333333,0.917973,0.625653,0.040728,0.799156,0.302898,None
2,42,1,svm,bsmote,0.50,0.405035,0.100000,0.071429,0.166667,0.984318,0.575493,0.056832,0.908926,0.252517,None
3,42,1,svm,ros,0.75,0.661723,0.053571,0.028302,0.500000,0.875754,0.687877,0.032149,0.778046,0.357647,None
4,42,1,svm,adasyn,0.50,0.552801,0.051948,0.028169,0.333333,0.916767,0.625050,0.044549,0.807600,0.302375,None
5,42,1,svm,casb,0.75,0.662178,0.054054,0.028571,0.500000,0.876960,0.688480,0.035481,0.801166,0.358116,None
6,42,1,svm,casb_relaxed,0.50,0.552437,0.051282,0.027778,0.333333,0.915561,0.624447,0.043502,0.812425,0.301860,None
7,42,1,svm,casb_edited,0.50,0.671225,0.065934,0.035294,0.500000,0.901086,0.700543,0.041254,0.817451,0.368579,None
8,42,1,svm,casb_ros,0.75,0.661267,0.053097,0.028037,0.500000,0.874548,0.687274,0.033631,0.792119,0.357182,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.016471,0.633092,0.000000,None
